# Practice Activity: Weather as a Business Covariate

**GSB 5544 · Computing and Machine Learning for Business Analytics**

You manage inventory planning for a beverage distributor with two markets: **San Luis Obispo, CA** and **Phoenix, AZ**. Before anyone builds a demand model, you need each market's weather history, and it lives in a bucket that is far larger than your laptop.

**Run this notebook inside your SageMaker Studio JupyterLab space.** Every code cell begins with a `# RUNS ON:` line that says where it executes. Everything you need is in the pre-class reading and the topics-of-practice notebook. Each question has a code cell; some also ask for a short written answer in a markdown cell. Q1 and Q11 need numbers you wrote down from the reading.

**Stations**

| Market | Station ID | Name |
|---|---|---|
| San Luis Obispo | `USC00047851` | SAN LUIS OBISPO POLY, CA |
| Phoenix | `USW00023183` | PHOENIX AIRPORT, AZ |

Run the first setup cell, then do **Part 1**, a short guided trip from the cloud to your laptop. Part 2 (the Athena setup cell and Q1 to Q13) follows it.

In [ ]:
# RUNS ON: SageMaker ml.t3.medium (us-east-1)
%pip install -q s3fs             # harmless if already present

import io
import json
import time
import datetime as dt
import psutil
import boto3
import pandas as pd
import matplotlib.pyplot as plt
from botocore import UNSIGNED
from botocore.config import Config

BUCKET   = "noaa-ghcn-pds"
REGION   = "us-east-1"
COLS     = ["id", "date", "element", "value", "m_flag", "q_flag", "s_flag", "obs_time"]
STATIONS = {"San Luis Obispo": "USC00047851", "Phoenix": "USW00023183"}

# unsigned: the public bucket does not care who is asking
s3 = boto3.client("s3", region_name=REGION, config=Config(signature_version=UNSIGNED))


def human(n, base=1000):
    for unit in ["B", "KB", "MB", "GB", "TB", "PB"]:
        if n < base:
            return f"{n:,.1f} {unit}"
        n /= base
    return f"{n:,.1f} EB"

## Part 1 · Guided workflow: from the cloud to your laptop

A full year of daily high temperatures for every weather station on Earth is 4.4 million rows. You will let a cloud machine read and shrink it, then carry the small result home and chart it on your own computer. Nothing here uses Athena or creates a bucket.

| Step | Where the code runs | Where the data is stored |
|---|---|---|
| 1. Sign in and open your space | nowhere yet | S3 (NOAA's public bucket) |
| 2. Read and summarize | **SageMaker** instance | S3 → SageMaker RAM → SageMaker disk |
| 3. Download the summary | no code; your browser copies a file | SageMaker disk → **your laptop's** disk |
| 4. Read and chart | **your laptop** | laptop disk → laptop RAM |
| 5. Save and stop | nowhere | your work stays on the SageMaker disk and your laptop |

### Step 1. Sign in and open your SageMaker space

*Code runs: nowhere yet. Data is stored: in S3.*

1. Go to [console.aws.amazon.com](https://console.aws.amazon.com) and sign in as **root user** with the email and password of the account you already created.
2. Check that the region, top right, reads **N. Virginia**.
3. Search for **SageMaker AI** and open it. In the left menu choose **SageMaker Studio**, then **Open Studio**.
4. In Studio click **JupyterLab**, then click `gsb5544` in the spaces table. Click **Run space**, wait for the status to read *Running*, then click **Open JupyterLab**. The meter is now on.
5. Upload this notebook with the upload arrow in the file browser, open it, and run the first setup cell above.

### Step 2. Read from S3 and summarize, on the cloud machine

*Code runs: on the SageMaker instance. Data is stored: in S3, then in this instance's RAM, then as a small file on this instance's disk.*

First, prove to yourself which machine this is.

In [ ]:
# RUNS ON: SageMaker ml.t3.medium (us-east-1)
import socket
print("This code is running on:", socket.gethostname())
print("RAM on this machine    :", round(psutil.virtual_memory().total / 1e9, 1), "GB")

Now read every station's 2024 daily highs straight from S3, keep California, and reduce it to one row per day. Fill in the three blanks.

In [ ]:
# RUNS ON: SageMaker ml.t3.medium (us-east-1)
t0 = time.perf_counter()
raw = pd.read_parquet(
    "s3://noaa-ghcn-pds/parquet/by_year/YEAR=2024/ELEMENT=TMAX/",
    columns=["ID", "DATE", "DATA_VALUE"],          # read only the columns we need
    storage_options={"anon": ____},              # unsigned: the bucket is public
)
print(f"Read {len(raw):,} rows from S3 in {time.perf_counter() - t0:.0f} s")
print(f"They occupy {human(raw.memory_usage(deep=True).sum())} of this machine's RAM")

ca = raw[raw["ID"].str.startswith("USC0004")].copy()          # California cooperative stations
ca["date"]   = pd.to_datetime(ca["DATE"].astype(str), format="%Y%m%d")
ca["tmax_c"] = ca["DATA_VALUE"] / ____                        # stored in tenths of a degree C

summary = (ca.groupby(____)["tmax_c"]
             .agg(stations="count", coolest="min", average="mean", hottest="max")
             .round(1)
             .reset_index())

summary.to_csv("ca_tmax_2024_daily.csv", index=False)          # written to THIS machine's disk
print(f"Summary: {len(summary)} rows, saved to the SageMaker disk")
summary.head()

You should see about 4.4 million rows read in a few seconds and a summary of 366 rows. The big table lives only in this machine's RAM and disappears when the space stops. The small file is on this machine's disk.

### Step 3. Download the small file to your computer

*Code runs: none. Data is stored: on the SageMaker disk, and after this step also on your laptop's disk.*

1. In the file browser on the left, click the refresh arrow. `ca_tmax_2024_daily.csv` appears next to this notebook.
2. Right-click it and choose **Download**. It lands in your laptop's Downloads folder. It is about 11 KB.

### Step 4. Read it and chart it, on your own computer

*Code runs: on your laptop. Data is stored: on your laptop's disk, then in your laptop's RAM.*

On your laptop, open Jupyter or VS Code, start a **new** notebook in the folder that holds the downloaded file, and paste this in. It is text here, not a cell, so that you cannot run it on the cloud machine by accident.

```python
# RUNS ON: laptop
import socket
import pandas as pd
import matplotlib.pyplot as plt

print("This code is running on:", socket.gethostname())

small = pd.read_csv("ca_tmax_2024_daily.csv", parse_dates=["date"])

fig, ax = plt.subplots(figsize=(9, 4))
ax.fill_between(small["date"], small["coolest"], small["hottest"], alpha=0.2,
                label="coolest to hottest station")
ax.plot(small["date"], small["average"], label="average of all stations")
ax.set_title("Daily high temperature across California stations, 2024")
ax.set_ylabel("°C")
ax.legend()
fig.savefig("ca_tmax_2024.png", dpi=150, bbox_inches="tight")
```

The computer name it prints should be your own, not the one from step 2. If `pandas` or `matplotlib` is missing, run `pip install pandas matplotlib` first.

### Step 5. Save your work and stop the cloud machine

*Code runs: nowhere. Data is stored: this notebook and the CSV on the SageMaker disk; the CSV, the chart, and your local notebook on your laptop.*

1. **Laptop:** save your local notebook. Keep `ca_tmax_2024.png`; you submit it with this notebook.
2. **SageMaker:** File → **Save Notebook**. Right-click this notebook in the file browser and **Download** a copy.
3. **Stop the instance.** If you are going straight on to Part 2, leave it running and stop it at Q13, which uses these same steps. Otherwise go to the Studio tab, open **JupyterLab**, click **Stop** on `gsb5544`, and wait for *Stopped*.

A stopped space keeps its disk, so your files are there next time. Its RAM is wiped, so `raw` and `summary` are gone and that is fine. You created nothing in S3, so there is nothing to delete there.

### Two questions

**P1. What happened in the cloud?** In two or three sentences, name which service *stored* the 4.4 million rows and which one *processed* them, and say why you did not have to pay to store them.

*Your answer:*

**P2. What happened on your own computer?** In two or three sentences, say what your laptop stored and processed, how large the file was that travelled between the two machines, and why that made the laptop step easy.

*Your answer:*

---

## Part 2 · The inventory questions

**Athena setup** (given; it is block J of the topics-of-practice notebook, unchanged). It creates your results bucket if needed, defines the helpers, and defines the `gsb5544.ghcn` table over the public Parquet files. DDL is free.

In [ ]:
# RUNS ON: SageMaker ml.t3.medium (us-east-1)
sts     = boto3.client("sts",    region_name=REGION)
athena  = boto3.client("athena", region_name=REGION)
s3_mine = boto3.client("s3",     region_name=REGION)

ACCOUNT        = sts.get_caller_identity()["Account"]
RESULTS_BUCKET = f"gsb5544-athena-{ACCOUNT}"
RESULTS        = f"s3://{RESULTS_BUCKET}/athena/"
s3_mine.create_bucket(Bucket=RESULTS_BUCKET)

ATHENA_BYTES = 0                                  # running total of everything scanned today; Q13 audits it

def run_athena(sql, database=None):
    """Start a query, wait for it, return (query id, statistics)."""
    global ATHENA_BYTES
    kwargs = {"QueryString": sql, "ResultConfiguration": {"OutputLocation": RESULTS}}
    if database:
        kwargs["QueryExecutionContext"] = {"Database": database}
    qid = athena.start_query_execution(**kwargs)["QueryExecutionId"]
    while True:
        q = athena.get_query_execution(QueryExecutionId=qid)["QueryExecution"]
        state = q["Status"]["State"]
        if state in ("SUCCEEDED", "FAILED", "CANCELLED"):
            break
        time.sleep(1)
    if state != "SUCCEEDED":
        raise RuntimeError(q["Status"].get("StateChangeReason", state))
    ATHENA_BYTES += q["Statistics"].get("DataScannedInBytes", 0)
    return qid, q["Statistics"]


def athena_df(qid):
    """Read a finished query's result CSV from the results bucket."""
    obj = s3_mine.get_object(Bucket=RESULTS_BUCKET, Key=f"athena/{qid}.csv")
    return pd.read_csv(io.BytesIO(obj["Body"].read()))


run_athena("CREATE DATABASE IF NOT EXISTS gsb5544")
run_athena('''
CREATE EXTERNAL TABLE IF NOT EXISTS gsb5544.ghcn (
    id string, `date` string, data_value bigint,
    m_flag string, q_flag string, s_flag string, obs_time string
)
PARTITIONED BY (year int, element string)
STORED AS PARQUET
LOCATION 's3://noaa-ghcn-pds/parquet/by_year/'
TBLPROPERTIES (
    'projection.enabled'        = 'true',
    'projection.year.type'      = 'integer',
    'projection.year.range'     = '1750,2030',
    'projection.element.type'   = 'enum',
    'projection.element.values' = 'TMAX,TMIN,PRCP,SNOW,SNWD,TAVG',
    'storage.location.template' = 's3://noaa-ghcn-pds/parquet/by_year/YEAR=${year}/ELEMENT=${element}/'
)
''', database="gsb5544")
print("Ready. Results bucket:", RESULTS_BUCKET)

## Set 1 · Capacity

### Q1. Two machines
Report physical cores, total RAM, available RAM, and free disk in GB for **this** machine (the SageMaker instance) and for **your laptop** (the numbers you wrote down from the reading, §1), as one DataFrame with a row per measure and a column per machine. Store this machine's available RAM in bytes as `MY_RAM`.

Then, in the markdown cell, one sentence: which machine would you rather use to read a 1.3 GB file from `noaa-ghcn-pds`, and why?

In [ ]:
# RUNS ON: SageMaker ml.t3.medium (us-east-1)
# your code here


*Your answer:*

### Q2. Rows that fit
A GHCN row is roughly 50 bytes on disk, and a text-heavy CSV needs about 3× its disk size once loaded in pandas. How many raw rows could this machine's available RAM hold? Print the number with thousands separators.

In [ ]:
# RUNS ON: SageMaker ml.t3.medium (us-east-1)
# your code here


### Q3. A decade of weather
List the by-year files for 2016 through 2025 and compute their **total** size. Print the total in readable units, then print how many copies of this machine's available RAM that is.

*Hint: a `Prefix` of `csv/by_year/201` returns 2010–2019; you will need two listings or a filter.*

In [ ]:
# RUNS ON: SageMaker ml.t3.medium (us-east-1)
# your code here


## Set 2 · Vocabulary in action

### Q4. What is under `csv/`?
List the common prefixes directly under `csv/` in the bucket. Then, in the markdown cell that follows, explain in one or two sentences why `by_year/` is not a folder and what S3 is actually doing when it shows you one.

In [ ]:
# RUNS ON: SageMaker ml.t3.medium (us-east-1)
# your code here


*Your answer:*

### Q5. Metadata only
Using a single call that does **not** download the file, print the size (readable) and the last-modified date of `csv/by_year/2024.csv`. Then print whether a plain `pd.read_csv` of it would fit in this machine's RAM at 3×.

In [ ]:
# RUNS ON: SageMaker ml.t3.medium (us-east-1)
# your code here


## Set 3 · Extraction

### Q6. Pull both markets
Write a function `load_station(station_id)` that:

1. prints the object's size before reading it,
2. checks whether the file has a header row using a range request,
3. reads the file keeping only `id`, `date`, `element`, `value`,
4. converts `date` to a datetime,
5. keeps only `TMAX` and `PRCP` rows from **2015 onward**,
6. returns the DataFrame.

Call it for both stations and combine the results into one DataFrame called `wx` with a `market` column.

This runs on the SageMaker instance: the bytes travel from S3 to this machine over AWS's network in the same region, and never touch your laptop.

In [ ]:
# RUNS ON: SageMaker ml.t3.medium (us-east-1)
# your code here


### Q7. Hot days per year
Temperatures are stored in tenths of a degree Celsius. For each market and year, count the number of days with a daily maximum of **35 °C or higher** (95 °F). Produce a table with years as rows and markets as columns.

In [ ]:
# RUNS ON: SageMaker ml.t3.medium (us-east-1)
# your code here


### Q8. Monthly rainfall
Precipitation is stored in tenths of a millimeter. For each market, compute total precipitation per month in millimeters, then show the **average for each calendar month** across all years (so 12 rows per market). Which month is wettest in each market?

In [ ]:
# RUNS ON: SageMaker ml.t3.medium (us-east-1)
# your code here


## Set 4 · Visualize and interpret

### Q9. One chart, two markets
Plot hot days per year (from Q7) for both markets on a single line chart. Label the axes with units and give the chart a title that says what it shows.

In [ ]:
# RUNS ON: SageMaker ml.t3.medium (us-east-1)
# your code here


### Q10. The business read
In five to seven sentences, answer as the inventory planner:

- What does the chart imply about how differently the two markets should be stocked across the year?
- Name **two** other datasets you would want to join to this weather history before trusting a demand model, and say where each one probably lives (your laptop, a company database, or a cloud bucket).
- This weather history will be refreshed **every week** once the model is live. Of the three places this module's code can run (your laptop, a SageMaker instance, the Athena fleet), which would you choose for that weekly refresh, and why? Answer from the reading now; revise after Q11 if the numbers change your mind.

*Your answer:*

## Set 5 · Where the work runs

### Q11. Three ways to get the same rows
Extract the **Phoenix `TMAX` and `PRCP` rows for 2024** three ways and time each one:

- **(a) Laptop.** You already did this before class: the timed chunked read of `csv/by_year/2024.csv` in the reading, §6. Paste your `LAPTOP_SECONDS` into the cell. (That read was for the San Luis Obispo station; same file, same amount of work.)
- **(b) SageMaker.** Run the same chunked read here, filtering each one-million-row chunk to the Phoenix station. Time it.
- **(c) Athena.** Query `gsb5544.ghcn` for the same rows. Time it and record the bytes scanned.

Build a DataFrame `race` with one row per tier and columns for seconds, bytes moved or scanned, and rows returned. Also confirm that (b) and (c) return the same row count as filtering `wx` to Phoenix and 2024.

In the markdown cell after, explain which tier you would use for this extraction and why, using the phrase "filter at the source." Say what changed between (a) and (b), and what changed between (b) and (c).

In [ ]:
# RUNS ON: SageMaker ml.t3.medium (us-east-1)
# your code here


In [ ]:
# RUNS ON: Athena fleet (results land in S3)
# your code here


*Your answer:*

### Q12. Hot days, straight from the fleet
Write **one Athena query** that returns the number of hot days (`TMAX` ≥ 35 °C, i.e. `data_value >= 350`) per station per year from 2015 onward, for both stations: the same table you built in Q7, but computed entirely on the fleet. Pivot the result to years as rows and markets as columns and check it against `hot`.

Then print two numbers side by side: the bytes Athena scanned for this query, and the bytes pandas read in Q6 (the two by-station CSVs).

In the markdown cell, explain the comparison. One of the two paths read fewer bytes. Which, and why? What does that tell you about when Athena over `by_year` is the right tool, and when choosing the right object (the by-station file) beats it?

In [ ]:
# RUNS ON: Athena fleet (results land in S3)
# your code here


*Your answer:*

### Q13. Cost audit and teardown (graded)
Every session on a rented machine ends the same way. Four parts; the last one is the evidence.

**(a) Empty the results bucket.** Print the number of objects and their total size before deleting, then confirm the bucket is empty.

**(b) Estimate today's spend.** Hours this space has been running × the `ml.t3.medium` hourly price, plus `ATHENA_BYTES` × the Athena price per TB (10 MB minimum per query). Print the two lines and the total. The cell reads the space's start time from SageMaker; if that lookup fails, set `hours` by hand from the time you clicked **Run space** (Learner Lab: from the lab timer).

**(c) Stop the space.** In Studio → JupyterLab spaces, click **Stop** on `gsb5544` and wait for *Stopped*. Then read **Credits remaining** on Console Home (Learner Lab: the budget on the lab page).

**(d) Evidence.** Either paste a screenshot into the markdown cell showing the space *Stopped* and the credits readout, or run the verification cell from your **laptop** (it needs AWS credentials in `~/.aws/credentials`; Learner Lab users paste the "AWS Details" block there first) and paste its output into the markdown cell.

In [ ]:
# RUNS ON: SageMaker ml.t3.medium (us-east-1)
# your code here


**Verification cell** for part (d). This one runs on your laptop *after* you have stopped the space, because a cell cannot stop the machine it is running on and then report on it.

In [ ]:
# RUNS ON: laptop
# Needs AWS credentials in ~/.aws/credentials (Learner Lab: paste the "AWS Details" block there first).
import boto3

sm = boto3.client("sagemaker", region_name="us-east-1")
apps = sm.list_apps()["Apps"]
for a in apps:
    print(f'{a["AppType"]:<12} {a.get("SpaceName", ""):<12} {a["Status"]}')

still_running = [a for a in apps if a["Status"] not in ("Deleted", "Failed")]
print("All stopped" if not still_running else "STILL RUNNING: go back to Studio and stop it")

*Evidence (screenshot or pasted output):*

---

**Submit** this notebook with all cells run inside SageMaker, plus your Q13 evidence. Your written answers for Q1, Q4, Q10, Q11, and Q12 go in the markdown cells provided. A submission is not complete while its JupyterLab space is still running; we check.